# 10 — Export geometry

Round-trip a part out to neutral formats so other CAD tools can
consume it. We'll write **STL** (for 3D printing) and **IGES**
(for general CAD interchange).

>  **Note (Alibre V29):** `ExportAP242` (STEP) currently access-
> violates on simple solids — a native crash that bypasses the CLR.
> We skip it here. Use IGES until upstream fixes it.

**Prereq:** open a fresh empty part in Alibre.

## Build something to export

In [ ]:
import os
import tempfile
from alibrex import (
    CurrentPart,
    ADDirectionType,
    ADPartFeatureEndCondition,
)

part = CurrentPart()

xy = part.DesignPlanes.Item(0)
sk = part.Sketches.AddSketch(None, xy, "Base")
figs = sk.Figures
figs.AddLine(0.0, 0.0, 3.0, 0.0)
figs.AddLine(3.0, 0.0, 3.0, 2.0)
figs.AddLine(3.0, 2.0, 0.0, 2.0)
figs.AddLine(0.0, 2.0, 0.0, 0.0)

part.Features.AddExtrudedBoss(
    sk, 1.0, ADPartFeatureEndCondition.AD_TO_DEPTH,
    None, None, 0.0,
    ADDirectionType.AD_ALONG_NORMAL, None, None, False,
    None, False,
    "Box", "Depth", "",
)

## Pick an output directory

In [ ]:
OUT_DIR = os.path.join(tempfile.gettempdir(), "alibrex_nb10")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_DIR

## Export STL

Arguments: `(path, deviation, angle_deg, max_edge_length)`. Smaller
deviation and angle mean finer tessellation.

In [ ]:
stl_path = os.path.join(OUT_DIR, "ExportPart.stl")
part.ExportSTL(stl_path, 0.5, 15.0, 0.05)
os.path.getsize(stl_path)

## Export IGES

In [ ]:
igs_path = os.path.join(OUT_DIR, "ExportPart.igs")
part.ExportIGES(igs_path)
os.path.getsize(igs_path)

## Summary

In [ ]:
for path in (stl_path, igs_path):
    size = os.path.getsize(path)
    print(f"  {os.path.basename(path):20s} {size:>10,} bytes")